# API 调用与检索

用设备说明试关键词检索和语义检索，再查看基于资料的回答。函数整理在 `main.py`。语义检索部分可以单独运行，不需要前面的 DeepSeek 客户端。

下方保留我保存的各次练习结果，按这里的打印格式整理。重新运行会更新输出，模型措辞可能不同。TXT部分保留从30天到45天的变化，失败结果也保留。

In [ ]:
from main import records, create_client, ask_model, find_records, build_context, answer_question

## 直接提问

输入密钥后修改问题试一下。运行下一格会请求 DeepSeek。

In [ ]:
client = create_client()
question = "Python列表和字典有什么区别？"
print(ask_model(question, None, client))

列表有序存多个值，用索引访问；字典存键值对，按键访问。


## 先看检索结果

检索返回记录列表，整理后得到保留来源的字符串。这格不请求模型。

In [ ]:
hits = find_records(records, "风机")
context = build_context(hits)
print(context)

风机说明.txt，第1段：每30天检查风机滤网。
风机说明.txt，第2段：风机过热时，应先停机，再检查风道。



## 根据找到的资料回答

分别检查：有答案、资料没写答案、没有相关资料。前两组会请求模型，最后一组直接返回。

In [ ]:
cases = [
    ("多久检查风机滤网？", "滤网"),
    ("风机的生产厂家是谁？", "风机"),
    ("电机的额定功率是多少？", "电机"),
]
for question, keyword in cases:
    print("问题：", question)
    print("回答：", answer_question(records, question, keyword, client))
    print()

问题： 多久检查风机滤网？
回答： 每30天检查风机滤网。（来源：风机说明.txt，第1段）

问题： 风机的生产厂家是谁？
回答： 原文未提供。

问题： 电机的额定功率是多少？
回答： 未检索到相关资料



## 换个关键词

‘过滤网’在原文中没有完整出现。这种查找无法识别它和‘滤网’的关系。

In [ ]:
print(find_records(records, "过滤网"))
assert find_records(records, "") == []
assert build_context([]) == ""

[]


## 语义检索

用 BGE 中文模型把四段原文编码成向量。首次运行会下载模型，使用 CPU；后面几格只检索，不请求 DeepSeek。

In [ ]:
from main import records, load_embedder, semantic_search, get_records

model = load_embedder()
texts = [record["text"] for record in records]
doc_vectors = model.encode(texts, normalize_embeddings=True)
print(doc_vectors.shape)

(4, 512)


### 用‘过滤网’找‘滤网’

问题和原文编码后比较余弦相似度。`corpus_id` 用来取回原始记录，`score` 用来排序。

In [ ]:
question = "过滤网多久维护一次？"
hits = semantic_search(records, question, model, doc_vectors, top_k=2)
for hit, record in zip(hits, get_records(records, hits)):
    print("来源：", record["source"])
    print("原文：", record["text"])
    print("相似度：", hit["score"])
    print()

来源： 风机说明.txt，第1段
原文： 每30天检查风机滤网。
相似度： 0.6077560186386108

来源： 水泵说明.txt，第1段
原文： 每90天检查水泵密封。
相似度： 0.4474352300167084



### 换三个问题

观察段落里是否包含答案。资料库非空时，取前两条通常仍会得到结果；相似度高不代表原文一定能回答问题。

In [ ]:
questions = [
    "机器太热应该怎么处理？",
    "水泵密封多久检查一次？",
    "设备的生产厂家是谁？",
]
for question in questions:
    hits = semantic_search(records, question, model, doc_vectors)
    print("问题：", question)
    for record in get_records(records, hits):
        print(record["source"], record["text"])
    print()

assert get_records(records, []) == []

问题： 机器太热应该怎么处理？
风机说明.txt，第2段 风机过热时，应先停机，再检查风道。
水泵说明.txt，第2段 水泵出现异常噪声时，应先停机检查。

问题： 水泵密封多久检查一次？
水泵说明.txt，第1段 每90天检查水泵密封。
水泵说明.txt，第2段 水泵出现异常噪声时，应先停机检查。

问题： 设备的生产厂家是谁？
水泵说明.txt，第2段 水泵出现异常噪声时，应先停机检查。
风机说明.txt，第1段 每30天检查风机滤网。



## 接通语义检索与模型回答

把匹配记录整理成带来源的文字，再交给 DeepSeek。匹配记录是包含字典的列表；参考文字和回答都是字符串。

In [ ]:
from main import records, load_embedder, semantic_find, build_context, create_client, answer_with_search

if "model" not in globals():
    model = load_embedder()
question = "过滤网多久维护一次？"
matched_records = semantic_find(records, question, model)
context = build_context(matched_records)
print(context)

风机说明.txt，第1段：每30天检查风机滤网。
水泵说明.txt，第1段：每90天检查水泵密封。



### 看原文，再看回答

下一格会请求 DeepSeek 一次。对照上面的原文，检查回答和来源；有检索结果也可能没有答案。

In [ ]:
if "client" not in globals():
    client = create_client()
print(answer_with_search(records, question, model, client))

根据风机说明.txt第1段，过滤网每30天检查一次。


### 换三个问题

下一格请求DeepSeek三次。保存的练习中，过热问题有一次返回“原文未提供”，保留这次失败；需要排查时可以单独打印 `build_context(semantic_find(records, question, model))` 查看候选资料。

In [ ]:
questions = [
    "机器太热应该怎么处理？",
    "水泵密封多久检查一次？",
    "设备的生产厂家是谁？",
]
for question in questions:
    print("问题：", question)
    print("回答：", answer_with_search(records, question, model, client))
    print()

问题： 机器太热应该怎么处理？
回答： 原文未提供。

问题： 水泵密封多久检查一次？
回答： 根据资料，水泵密封每90天检查一次。来源：水泵说明.txt，第1段：“每90天检查水泵密封。”

问题： 设备的生产厂家是谁？
回答： 原文未提供。



### 提前结束的情况

空资料和空问题直接返回提示，这格不请求模型。

In [ ]:
print(answer_with_search([], "多久检查滤网？", model, client))
print(answer_with_search(records, "", model, client))

未检索到相关资料
请输入问题


## 从 TXT 读取资料

两份模拟说明，每个非空行是一条记录。来源保留文件路径和原始行号。已有文件会保留，方便修改后重新读取。

In [ ]:
from pathlib import Path
from main import load_records, load_embedder, semantic_find, build_context, create_client, answer_with_search

document_dir = Path("practice_documents05")
document_dir.mkdir(exist_ok=True)
file_contents = {
    "风机说明.txt": "风机A每30天检查一次滤网。\n\n风机A过热时，应先停机，再检查风道。\n风机A启动前应确认防护罩安装牢固。\n",
    "水泵说明.txt": "水泵B每90天检查一次密封。\n水泵B出现异常噪声时，应先停机检查。\n\n水泵B首次投入运行前，应确认管路连接完好。\n",
}
for name, content in file_contents.items():
    path = document_dir / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")
filenames = [document_dir / name for name in file_contents]
file_records = load_records(filenames)
print("记录数量：", len(file_records))
for record in file_records:
    print(record)
print("空文件列表：", load_records([]))

记录数量： 6
{'text': '风机A每30天检查一次滤网。', 'source': 'practice_documents05\\风机说明.txt，第1行'}
{'text': '风机A过热时，应先停机，再检查风道。', 'source': 'practice_documents05\\风机说明.txt，第3行'}
{'text': '风机A启动前应确认防护罩安装牢固。', 'source': 'practice_documents05\\风机说明.txt，第4行'}
{'text': '水泵B每90天检查一次密封。', 'source': 'practice_documents05\\水泵说明.txt，第1行'}
{'text': '水泵B出现异常噪声时，应先停机检查。', 'source': 'practice_documents05\\水泵说明.txt，第2行'}
{'text': '水泵B首次投入运行前，应确认管路连接完好。', 'source': 'practice_documents05\\水泵说明.txt，第4行'}
空文件列表： []


### 查看文件里的候选资料

这格只检索，首次运行可能下载模型，不请求 DeepSeek。

In [ ]:
if "model" not in globals():
    model = load_embedder()
question = "风机A滤网多久检查一次？"
print(build_context(semantic_find(file_records, question, model)))

practice_documents05\风机说明.txt，第1行：风机A每30天检查一次滤网。
practice_documents05\风机说明.txt，第3行：风机A过热时，应先停机，再检查风道。



### 用文件中的原文回答

下一格请求 DeepSeek 三次。对照来源文件核对回答。

In [ ]:
if "client" not in globals():
    client = create_client()
for question in ["风机A滤网多久检查一次？", "风机A过热时应该怎么处理？", "风机A的生产厂家是谁？"]:
    print("问题：", question)
    print("回答：", answer_with_search(file_records, question, model, client))
    print()

问题： 风机A滤网多久检查一次？
回答： 风机A每30天检查一次滤网。  
来源：practice_documents05\风机说明.txt，第1行。

问题： 风机A过热时应该怎么处理？
回答： 风机A过热时，应先停机，再检查风道。（来源：practice_documents05\风机说明.txt，第3行）

问题： 风机A的生产厂家是谁？
回答： 原文未提供。



### 修改文件后重新读取

在 VS Code 打开下面打印的文件，把第一行的30改为45并保存，然后运行最后一格。TXT 已改为45天；原读取练习保留了30天的旧输出，后面的固定测试已得到45天回答。

In [ ]:
print(filenames[0].resolve())

D:\1rance\LLM\practice_documents05\风机说明.txt


下面会请求 DeepSeek 一次。先看重新读取的第一条原文是否已变成45天，再核对回答；旧列表不会随着文件修改自动更新。

In [ ]:
file_records = load_records(filenames)
print("重新读取的第一条原文：", file_records[0]["text"])
print(answer_with_search(file_records, "风机A滤网多久检查一次？", model, client))

重新读取的第一条原文： 风机A每45天检查一次滤网。
风机A每45天检查一次滤网。  
来源：practice_documents05\风机说明.txt，第1行。


## 固定测试与结果记录

六个问题覆盖周期、换一种说法、过热处理、错误前提和资料缺失。先根据原文写预期，预期只供核对，不发送给模型。下面保存的输出来自本次练习与结果 JSON，重跑时会更新。

In [ ]:
from main import test_cases, ask_and_record

file_records = load_records(filenames)
if not file_records or file_records[0]["text"] != "风机A每45天检查一次滤网。":
    raise ValueError("请先把风机TXT第一行改为45天并保存，再重新读取。")
for case in test_cases:
    print(case["id"], case["question"], "预期：", case["expected"])

T01 风机A滤网多久检查一次？ 预期： 45天；依据风机说明.txt第1行。
T02 风机A的过滤网多久维护一次？ 预期： 45天；检查换一种说法是否仍能找到滤网资料，依据风机说明.txt第1行。
T03 风机A过热时应该怎么处理？ 预期： 先停机，再检查风道；依据风机说明.txt第3行。
T04 水泵B密封多久检查一次？ 预期： 90天；依据水泵说明.txt第1行。
T05 风机A的滤网是不是每30天检查一次？ 预期： 应纠正为45天，不能赞同30天；依据风机说明.txt第1行。
T06 风机A的生产厂家是谁？ 预期： 原文未提供；不能编造厂家信息。


### 收集六条结果

下一格会请求 DeepSeek 六次。每条结果保留问题、预期、检索记录、参考文字和回答；失败结果也保留。

In [ ]:
results = []
for case in test_cases:
    result = ask_and_record(file_records, case, model, client)
    results.append(result)
    print("编号：", result["id"])
    print("问题：", result["question"])
    print("回答：", result["answer"])
    print()
print("结果数量：", len(results))

编号： T01
问题： 风机A滤网多久检查一次？
回答： 风机A每45天检查一次滤网。  
来源：practice_documents05\风机说明.txt，第1行。

编号： T02
问题： 风机A的过滤网多久维护一次？
回答： 风机A的过滤网每45天检查一次。  
来源：practice_documents05\风机说明.txt，第1行：风机A每45天检查一次滤网。

编号： T03
问题： 风机A过热时应该怎么处理？
回答： 风机A过热时，应先停机，再检查风道。  
来源：practice_documents05\风机说明.txt，第3行。

编号： T04
问题： 水泵B密封多久检查一次？
回答： 水泵B每90天检查一次密封。  
（来源：practice_documents05\水泵说明.txt，第1行：水泵B每90天检查一次密封。）

编号： T05
问题： 风机A的滤网是不是每30天检查一次？
回答： 不是。根据资料，风机A的滤网是每45天检查一次。（来源：practice_documents05\风机说明.txt，第1行：风机A每45天检查一次滤网。）

编号： T06
问题： 风机A的生产厂家是谁？
回答： 原文未提供。

结果数量： 6


### 核对错误前提与资料缺失

对照 T05、T06 的检索原文和回答，填写自己的判断。保存的本次结果中，这两条已写了审查理由；重跑后需要重新核对再填写，留空表示未审查。

In [ ]:
for index in [4, 5]:
    result = results[index]
    print(result["id"], result["question"])
    print("检索原文：", result["retrieved"])
    print("回答：", result["answer"])
    print()

# 核对本次新回答后填写“通过/不通过 + 依据”，再保存。
results[4]["review"] = ""
results[5]["review"] = ""

T05 风机A的滤网是不是每30天检查一次？
检索原文： [{'text': '风机A每45天检查一次滤网。', 'source': 'practice_documents05\\风机说明.txt，第1行'}, {'text': '水泵B每90天检查一次密封。', 'source': 'practice_documents05\\水泵说明.txt，第1行'}]
回答： 不是。根据资料，风机A的滤网是每45天检查一次。（来源：practice_documents05\风机说明.txt，第1行：风机A每45天检查一次滤网。）

T06 风机A的生产厂家是谁？
检索原文： [{'text': '风机A启动前应确认防护罩安装牢固。', 'source': 'practice_documents05\\风机说明.txt，第4行'}, {'text': '风机A过热时，应先停机，再检查风道。', 'source': 'practice_documents05\\风机说明.txt，第3行'}]
回答： 原文未提供。



### 保存为 JSON

以下保留的是原练习保存的六条完整记录，包括 T05、T06 的两条判断。重新运行本单元会保存当前 `results`，不会自动判定通过。生成的 JSON 留在本地，完整示例输出保留在这里。

In [ ]:
import json

with open("llm06_test_results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

[
  {
    "id": "T01",
    "question": "风机A滤网多久检查一次？",
    "expected": "45天；依据风机说明.txt第1行。",
    "retrieved": [
      {
        "text": "风机A每45天检查一次滤网。",
        "source": "practice_documents05\\风机说明.txt，第1行"
      },
      {
        "text": "风机A过热时，应先停机，再检查风道。",
        "source": "practice_documents05\\风机说明.txt，第3行"
      }
    ],
    "context": "practice_documents05\\风机说明.txt，第1行：风机A每45天检查一次滤网。\npractice_documents05\\风机说明.txt，第3行：风机A过热时，应先停机，再检查风道。\n",
    "answer": "风机A每45天检查一次滤网。  \n来源：practice_documents05\\风机说明.txt，第1行。"
  },
  {
    "id": "T02",
    "question": "风机A的过滤网多久维护一次？",
    "expected": "45天；依据风机说明.txt第1行。检查换一种说法是否仍能找到滤网资料。",
    "retrieved": [
      {
        "text": "风机A每45天检查一次滤网。",
        "source": "practice_documents05\\风机说明.txt，第1行"
      },
      {
        "text": "风机A启动前应确认防护罩安装牢固。",
        "source": "practice_documents05\\风机说明.txt，第4行"
      }
    ],
    "context": "practice_documents05\\风机说明.txt，第1行：风机A每45天检查一次滤网。\npractice_documents05\\风机说明.txt，第4行：风机A启动前应确认防

## 公开手册实验

我把流程用到真实PDF，运行15个问题并比较取1块和3块。完整运行在[公开UPS手册Notebook](public_manual/demo.ipynb)。

下面直接读取保存的实际结果，不请求模型。R03、R04、R09保留了没有找到具体依据的失败回答；`status=ok`表示请求完成。

In [ ]:
import json
from pathlib import Path

manual_results = json.loads(Path("public_manual/results/llm07_all15.json").read_text(encoding="utf-8"))
for result in manual_results:
    print(result["id"], result["question"])
    print(result["answer"])
    print()
print("结果数量：", len(manual_results))

R01 手册中输出电压的出厂默认设置是多少？
输出电压的出厂默认设置是 230 VAC。来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01]。

R02 输出电压允许选择哪些值？
输出电压允许选择 220、230、240 VAC。  
依据：“输出电压 230 VAC 220、230、240 VAC 允许用户在 UPS 在线操作时选择输出电压。”来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01。

R03 声音报警出厂时是开启还是关闭？
资料未提供。摘录仅提到“当实际运行时间达到用户设置的限制时，UPS 将发出声音警报”（PDF第15页，块03），并说明“仅当 UPS 在电池模式下工作时声音警报才会发出”，未说明声音报警的出厂默认开/关状态。

R04 绿色或高效率模式默认开启吗？
资料未提供。摘录中没有关于“绿色或高效率模式”默认开启状态的说明。

R05 输出电压的出厂默认设置是不是110V？
不是。输出电压的出厂默认设置是 230 VAC，用户可选择 220、230、240 VAC。  
来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01。

R06 手册说PowerChute可以安装在32位系统上吗？
不是。手册明确说明 PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上：“注：PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03；同页块02也有相同说明）

R07 RS232和USB是否可以同时用于通信？
不可以。原句：“注：RS232 和 USB 无法同时使用。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03）

R08 手册给出的PowerChute软件下载网址是什么？
手册给出的下载网址是：`www.apc.com/pcss`。依据原文：“请登录 www.apc.com/pcss 下载并安装最新版PowerChute Serial Shutdown。”来源：EasyUPS_SRV_Chinese_990-91063B

### 取1块与3块的已保存对比

两组回答都有依据。参考文字从250字增加到872字，这个样例没有看到回答改善。下一格只读取保存文件。

In [ ]:
comparison = json.loads(Path("public_manual/results/llm07_topk_comparison.json").read_text(encoding="utf-8"))
for result in comparison:
    print("top_k：", result["top_k"], "参考文字长度：", result["context_characters"])
    print(result["answer"])
    print()

top_k： 1 参考文字长度： 250
不可以。手册说明：“PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03）

top_k： 3 参考文字长度： 872
不可以。手册明确说：“PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块02/块03）

